# 03 · Clasificación de Consultas
Identifica la **intención** del usuario y determina si la consulta requiere
acceso al corpus o puede responderse directamente.

### Categorías
| Categoría | Descripción |
|---|---|
| `búsqueda` | Solicita hechos/datos específicos del corpus |
| `resumen` | Pide resumir uno o varios papers |
| `comparación` | Quiere contrastar documentos/conceptos del corpus |
| `general` | Pregunta general sin necesidad de corpus |

### ¿Por qué Gemini aquí?
La clasificación exige **comprensión contextual profunda**: distinguir
matices entre "buscar" y "resumir", detectar referencias implícitas a
documentos y razonar sobre la intención. Gemini 2.0 Flash sobresale en
seguimiento de instrucciones complejas y razonamiento semántico.


In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────
!pip install -q \
    langchain==0.3.14 \
    langchain-google-genai==2.0.8 \
    pydantic==2.10.4
print('✓ Dependencias instaladas')


In [ ]:
from google.colab import userdata

GEMINI_KEY   = userdata.get('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.0-flash'


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ClasificacionConsulta(BaseModel):
    """Esquema de salida estructurada del clasificador de consultas."""
    categoria: Literal['búsqueda', 'resumen', 'comparación', 'general'] = Field(
        description='Tipo de consulta identificado'
    )
    requiere_rag: bool = Field(
        description='True si necesita búsqueda en la base vectorial'
    )
    doc_ids_mencionados: list[str] = Field(
        default=[],
        description='IDs de documentos mencionados explícitamente (e.g. doc_01)'
    )
    razonamiento: str = Field(
        description='Explicación breve de la clasificación'
    )


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

def crear_clasificador(api_key: str):
    """
    Construye la cadena LangChain para clasificar consultas del usuario.

    Usa Gemini 2.0 Flash con temperatura=0 para resultados deterministas.
    La cadena = Prompt | LLM | PydanticOutputParser.

    Args:
        api_key : Google Gemini API key.

    Returns:
        Función `clasificar(query: str) -> ClasificacionConsulta`.
    """
    llm = ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        google_api_key=api_key,
        temperature=0.0
    )
    parser = PydanticOutputParser(pydantic_object=ClasificacionConsulta)

    prompt = ChatPromptTemplate.from_messages([
        ('system', """Eres el clasificador de consultas de un sistema RAG biomédico.
El corpus contiene 50 papers científicos de arXiv sobre bioinformática,
genómica, diagnóstico médico con IA y estructuras proteicas.

Reglas:
- búsqueda  → el usuario pide datos/hechos específicos de los papers
- resumen   → el usuario pide resumir uno o varios papers del corpus
- comparación → el usuario quiere contrastar métodos, papers o enfoques
- general   → pregunta de conocimiento general (no requiere corpus)

{format_instructions}"""),
        ('human', 'Consulta: {query}')
    ])

    chain = prompt | llm | parser

    def clasificar(query: str) -> ClasificacionConsulta:
        """Clasifica la consulta y retorna estructura con categoría y metadatos."""
        return chain.invoke({
            'query': query,
            'format_instructions': parser.get_format_instructions()
        })

    return clasificar


In [ ]:
# ── Crear clasificador y ejecutar pruebas ─────────────────────────────────
clasificar = crear_clasificador(GEMINI_KEY)

consultas_test = [
    '¿Qué métodos de deep learning se usan para predecir la estructura de proteínas?',
    'Por favor resume el paper doc_01',
    'Compara el enfoque de doc_03 y doc_07 para el diagnóstico de cáncer',
    '¿Qué es el ADN?',
    'Dame un resumen de los papers sobre drug discovery del corpus',
    '¿Cuáles son los principales datasets de bioinformática usados en ML?',
    'Busca papers sobre redes neuronales convolucionales en imágenes médicas',
    '¿Cuánto es 2 + 2?',
]

print('=== PRUEBAS DE CLASIFICACIÓN ===\n')
resultados_clasificacion = []
for consulta in consultas_test:
    r = clasificar(consulta)
    resultados_clasificacion.append({'query': consulta, 'resultado': r.model_dump()})
    print(f'Consulta   : {consulta[:70]}')
    print(f'Categoría  : {r.categoria}')
    print(f'Req. RAG   : {r.requiere_rag}')
    print(f'Razonam.   : {r.razonamiento[:100]}')
    if r.doc_ids_mencionados:
        print(f'Docs citados: {r.doc_ids_mencionados}')
    print()
